In [1]:
import zipfile
import hashlib
import pandas as pd
from pathlib import Path
import re

# Define the root directory for reproducible test results
root_dir = Path("./packer/results/").resolve()
build_aab_path = (
    "build/outputs/bundle/playProdRelease/Signal-Android-play-prod-release.aab"
)

In [2]:
# compute the SHA-256 hash of a file
def sha256_file(path: Path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
            h.update(chunk)
    return h.hexdigest()


# extract .dex files from an AAB to a 'dex/' folder
def extract_dex_files(aab_path: Path, extract_to):
    with zipfile.ZipFile(aab_path, "r") as zipf:
        for file in zipf.namelist():
            if file.endswith(".dex"):
                zipf.extract(file, path=extract_to)

## `v7.41.3` data

### Initial look

In [3]:
# Initialize list to collect all data
records = []

# Loop through each test run folder
for run_dir in root_dir.rglob("*7_41*"):
    if not run_dir.is_dir():
        continue

    run_name = run_dir.name
    aab_path = run_dir / "bundle.aab"
    build_aab = run_dir / build_aab_path
    bundle_dex_dir = run_dir / "dex"
    bundle_dex_dir.mkdir(exist_ok=True)

    # Step 1: Unzip .dex files into 'dex/' folder
    extract_dex_files(aab_path, bundle_dex_dir)

    # Step 2: Assert hash of bundle.aab matches the reference one
    if build_aab.exists():
        bundle_hash = sha256_file(aab_path)
        reference_hash = sha256_file(build_aab)
        assert bundle_hash == reference_hash, f"Hash mismatch in {run_name}"

    # Step 3: Hash each dex file in 'dex/' and in reference build
    # 3a: From bundle
    for dex_file in bundle_dex_dir.rglob("*.dex"):
        records.append(
            {
                "run_name": run_name,
                "dex_name": dex_file.name,
                "hash": sha256_file(dex_file),
                "from_bundle": True,
            }
        )

    # 3b: From build
    build_dex_folder = (
        run_dir / "build/intermediates/dex/playProdRelease/minifyPlayProdReleaseWithR8/"
    )
    count = 1
    for dex_file in build_dex_folder.rglob("*.dex"):
        if dex_file.exists():
            records.append(
                {
                    "run_name": run_name,
                    "dex_name": dex_file.name,
                    "hash": sha256_file(dex_file),
                    "from_bundle": False,
                }
            )
            count += 1
    classes_1000_dex = (
        run_dir
        / "build/intermediates/desugar_lib_dex/playProdRelease/l8DexDesugarLibPlayProdRelease/classes1000.dex"
    )
    if classes_1000_dex.exists():
        fname = f"classes{count}.dex"
        print(f"[INFO] Desugared libs dex exists, saving it as {fname}")
        records.append(
            {
                "run_name": run_name,
                "dex_name": fname,
                "hash": sha256_file(classes_1000_dex),
                "from_bundle": False,
            }
        )
# Step 4: Create dataframe
df = pd.DataFrame(records)
df.set_index(["run_name", "from_bundle"], inplace=True)
df.sort_index(inplace=True)

[INFO] Desugared libs dex exists, saving it as classes7.dex
[INFO] Desugared libs dex exists, saving it as classes7.dex
[INFO] Desugared libs dex exists, saving it as classes7.dex
[INFO] Desugared libs dex exists, saving it as classes7.dex
[INFO] Desugared libs dex exists, saving it as classes7.dex
[INFO] Desugared libs dex exists, saving it as classes7.dex
[INFO] Desugared libs dex exists, saving it as classes7.dex
[INFO] Desugared libs dex exists, saving it as classes7.dex
[INFO] Desugared libs dex exists, saving it as classes7.dex
[INFO] Desugared libs dex exists, saving it as classes7.dex
[INFO] Desugared libs dex exists, saving it as classes7.dex
[INFO] Desugared libs dex exists, saving it as classes7.dex


In [4]:
df.loc["signal_7_41_3__dfs-sort__run_5"]

,dex_name,hash
from_bundle,,
False,classes.dex,ab85b85150775d1b4bf7f1f88fbf7f8bab9f051a7faa99...
False,classes2.dex,6bced746ee85eac654b2fcb64c2950695993abf0704275...
False,classes3.dex,b10f00c2d4d39f7e25c4bea7a807d5794472256625e778...
False,classes4.dex,c4a3f030950646989feb0df8aab55dadcf085c509796ab...
False,classes5.dex,e3fc2ead7fe95e874d3865e4f69e3026b2cc5cb76a691e...
False,classes6.dex,9834e7af7b38acd580bd91d0e3ef543f65a46fb0988aba...
False,classes7.dex,55cb98b238d8d8a1e12ebe1a30b86d477f055c7b8aa176...
True,classes.dex,ab85b85150775d1b4bf7f1f88fbf7f8bab9f051a7faa99...
True,classes2.dex,55cb98b238d8d8a1e12ebe1a30b86d477f055c7b8aa176...


### Basic asseert, checking that dex_files have not been altered after the bundle got created

In [5]:
for run_name, group in df.groupby(level="run_name"):
    try:
        print(f"[{run_name}]")
        # Split into bundle and non-bundle
        bundle_df = group.loc[(run_name, True)].set_index("dex_name")
        build_df = group.loc[(run_name, False)].set_index("dex_name")

        # Make sure both have the same dex files
        assert set(bundle_df.index) == set(
            build_df.index
        ), f"[{run_name}] Mismatched dex files"

        # Compare hashes
        for dex_name in bundle_df.index:
            hash_bundle = bundle_df.loc[dex_name, "hash"]
            hash_build = build_df.loc[dex_name, "hash"]
            if hash_bundle != hash_build:
                print(
                    f"[{run_name}] Hash mismatch in {dex_name} (build: {hash_build[:8]} / bundle: {hash_bundle[:8]})"
                )
    except KeyError:
        print(f"[{run_name}] Missing either bundle or build entries — skipping.")

[signal_7_41_3__dfs-sort__chaos_1]
[signal_7_41_3__dfs-sort__chaos_1] Hash mismatch in classes2.dex (build: 6bced746 / bundle: 55cb98b2)
[signal_7_41_3__dfs-sort__chaos_1] Hash mismatch in classes3.dex (build: b10f00c2 / bundle: 6bced746)
[signal_7_41_3__dfs-sort__chaos_1] Hash mismatch in classes4.dex (build: c4a3f030 / bundle: b10f00c2)
[signal_7_41_3__dfs-sort__chaos_1] Hash mismatch in classes5.dex (build: e3fc2ead / bundle: c4a3f030)
[signal_7_41_3__dfs-sort__chaos_1] Hash mismatch in classes6.dex (build: 9834e7af / bundle: e3fc2ead)
[signal_7_41_3__dfs-sort__chaos_1] Hash mismatch in classes7.dex (build: 55cb98b2 / bundle: 9834e7af)
[signal_7_41_3__dfs-sort__chaos_2]
[signal_7_41_3__dfs-sort__chaos_2] Hash mismatch in classes2.dex (build: 6bced746 / bundle: 55cb98b2)
[signal_7_41_3__dfs-sort__chaos_2] Hash mismatch in classes3.dex (build: b10f00c2 / bundle: 6bced746)
[signal_7_41_3__dfs-sort__chaos_2] Hash mismatch in classes4.dex (build: c4a3f030 / bundle: b10f00c2)
[signal_7_41

Note: there was an assumption that `classesN.dex` represents the desuragered libraries (`classes1000.dex` in the intermediate build files). This was wrong hovwever, which along with a bad computation of the associated sha256, led to the following block.

> ~~We can see that the dex files with the same name get altered between intermediates creation and their addition to the app bundle. **Notably `classes.dex` is repeatable across all runs**, and doesn't seem to be altered further after creation.~~

~~This makes sense, being supported also by the inclussion of the `classes1000.dex` (renamed to `classes7.dex` in the bundle), which contains desugared libraries (Java8 apis).~~

~~Funny enough, those files do not seem to be altered further than a filename change. For the other dex files, the change seems to be repeatable across runs, the file `classes{i}.dex` in build (`from_bundle=False`) maps to `classes_{i+1}` in bundle (`from_bundle=True`).~~

~~This leaves an unexplained difference between `classes2.dex` in bundle and `classes6.dex` in build.~~

In [6]:
import pandas as pd
import re


def remap_dex_name(dex_name, max_index):
    # If it's the base class (classes.dex), leave it unchanged
    if dex_name == "classes.dex":
        return dex_name

    # Check if it's in the form of classes{i}.dex
    match = re.match(r"classes(\d+)\.dex", dex_name)
    if match:
        i = int(match.group(1))  # Extract the number from 'classes{i}.dex'

        # If it's the last file, remap to classes2.dex
        if i == max_index:
            return "classes2.dex"

        # Otherwise, increment the class index (e.g., classes1.dex -> classes2.dex)
        if i < max_index:
            return f"classes{i + 1}.dex"

    # If it doesn't match any expected pattern, return the original name (fallback)
    return dex_name


def remap_dex_names(df):
    # Create a copy of the original DataFrame to avoid modifying in place
    new_df = df.copy()

    # Iterate through each 'run_name' group
    for run_name, group in new_df.groupby(level="run_name"):
        # Filter the rows where 'from_bundle' is False (for the specific 'run_name')
        build_df = group[group.index.get_level_values("from_bundle") == False]

        # Find the max index from dex_name (i.e., the highest number in 'classes{i}.dex')
        max_index = 0
        for dex_name in build_df["dex_name"].unique():
            match = re.match(r"classes(\d+)\.dex", dex_name)
            if match:
                i = int(match.group(1))
                max_index = max(max_index, i)

        # Apply the remapping to the 'dex_name' column in the build_df
        for idx, row in build_df.iterrows():
            new_dex_name = remap_dex_name(row["dex_name"], max_index)
            # Update the 'dex_name' in the new_df copy
            new_df.loc[idx, "dex_name"] = new_dex_name

    return new_df


# Remap dex_name based on the rules
remapped_df = remap_dex_names(df)

In [7]:
remapped_df.loc["signal_7_41_3__dfs-sort__run_5"]

,dex_name,hash
from_bundle,,
False,classes2.dex,ab85b85150775d1b4bf7f1f88fbf7f8bab9f051a7faa99...
False,classes2.dex,6bced746ee85eac654b2fcb64c2950695993abf0704275...
False,classes2.dex,b10f00c2d4d39f7e25c4bea7a807d5794472256625e778...
False,classes2.dex,c4a3f030950646989feb0df8aab55dadcf085c509796ab...
False,classes2.dex,e3fc2ead7fe95e874d3865e4f69e3026b2cc5cb76a691e...
False,classes2.dex,9834e7af7b38acd580bd91d0e3ef543f65a46fb0988aba...
False,classes2.dex,55cb98b238d8d8a1e12ebe1a30b86d477f055c7b8aa176...
True,classes.dex,ab85b85150775d1b4bf7f1f88fbf7f8bab9f051a7faa99...
True,classes2.dex,55cb98b238d8d8a1e12ebe1a30b86d477f055c7b8aa176...


In [8]:
for run_name, group in df.groupby(level="run_name"):
    try:
        print(f"[{run_name}]")
        # Split into bundle and non-bundle
        bundle_df = group.loc[(run_name, True)].set_index("dex_name")
        build_df = group.loc[(run_name, False)].set_index("dex_name")

        # Make sure both have the same dex files
        assert set(bundle_df.index) == set(
            build_df.index
        ), f"[{run_name}] Mismatched dex files"

        # Compare hashes
        for dex_name in bundle_df.index:
            hash_bundle = bundle_df.loc[dex_name, "hash"]
            hash_build = build_df.loc[dex_name, "hash"]
            if hash_bundle != hash_build:
                print(
                    f"[{run_name}] Hash mismatch in {dex_name} (build: {hash_build[:8]} / bundle: {hash_bundle[:8]})"
                )
    except KeyError:
        print(f"[{run_name}] Missing either bundle or build entries — skipping.")

[signal_7_41_3__dfs-sort__chaos_1]
[signal_7_41_3__dfs-sort__chaos_1] Hash mismatch in classes2.dex (build: 6bced746 / bundle: 55cb98b2)
[signal_7_41_3__dfs-sort__chaos_1] Hash mismatch in classes3.dex (build: b10f00c2 / bundle: 6bced746)
[signal_7_41_3__dfs-sort__chaos_1] Hash mismatch in classes4.dex (build: c4a3f030 / bundle: b10f00c2)
[signal_7_41_3__dfs-sort__chaos_1] Hash mismatch in classes5.dex (build: e3fc2ead / bundle: c4a3f030)
[signal_7_41_3__dfs-sort__chaos_1] Hash mismatch in classes6.dex (build: 9834e7af / bundle: e3fc2ead)
[signal_7_41_3__dfs-sort__chaos_1] Hash mismatch in classes7.dex (build: 55cb98b2 / bundle: 9834e7af)
[signal_7_41_3__dfs-sort__chaos_2]
[signal_7_41_3__dfs-sort__chaos_2] Hash mismatch in classes2.dex (build: 6bced746 / bundle: 55cb98b2)
[signal_7_41_3__dfs-sort__chaos_2] Hash mismatch in classes3.dex (build: b10f00c2 / bundle: 6bced746)
[signal_7_41_3__dfs-sort__chaos_2] Hash mismatch in classes4.dex (build: c4a3f030 / bundle: b10f00c2)
[signal_7_41

## Bugfix and secondary check:

In [18]:
# Helper function to remap dex_name based on your rules
def remap_dex_name(dex_name, max_index):
    # If it's the base class (classes.dex), leave it unchanged
    if dex_name == "classes.dex":
        return dex_name
    # Check if it's in the form of classes{i}.dex
    match = re.match(r"classes(\d+)\.dex", dex_name)
    if match:
        i = int(match.group(1))
        if i < max_index:
            return f"classes{i+1}.dex"  # Remap classes{i}.dex to classes{i+1}.dex
        elif i == max_index:
            return "classes2.dex"  # Last class (classes{N}.dex) becomes classes2.dex
    return dex_name  # If it's something else, return unchanged


# Initialize list to collect all data
records = []

# Loop through each test run folder
for run_dir in root_dir.rglob("*7_41*"):
    if not run_dir.is_dir():
        continue

    run_name = run_dir.name
    aab_path = run_dir / "bundle.aab"
    build_aab = run_dir / build_aab_path
    bundle_dex_dir = run_dir / "dex"
    bundle_dex_dir.mkdir(exist_ok=True)

    # Step 1: Unzip .dex files into 'dex/' folder
    extract_dex_files(aab_path, bundle_dex_dir)

    # Step 2: Assert hash of bundle.aab matches the reference one
    if build_aab.exists():
        bundle_hash = sha256_file(aab_path)
        reference_hash = sha256_file(build_aab)
        assert bundle_hash == reference_hash, f"Hash mismatch in {run_name}"

    # Step 3: Hash each dex file in 'dex/' and in reference build
    # 3a: From bundle
    for dex_file in bundle_dex_dir.rglob("*.dex"):
        records.append(
            {
                "run_name": run_name,
                "dex_name": dex_file.name,
                "hash": sha256_file(dex_file),
                "from_bundle": True,
            }
        )

    # 3b: From build
    build_dex_folder = (
        run_dir / "build/intermediates/dex/playProdRelease/minifyPlayProdReleaseWithR8/"
    )
    build_dex_files = list(build_dex_folder.rglob("*.dex"))

    # Initialize counter for renaming dex files
    count = 1

    # Process the build dex files
    for dex_file in build_dex_files:
        if dex_file.exists():
            # For all classes{i}.dex files, increment and rename sequentially
            if dex_file.name != "classes.dex":
                remapped_name = f"classes{count + 1}.dex"  # Remap to classes{i+1}.dex
                records.append(
                    {
                        "run_name": run_name,
                        "dex_name": remapped_name,
                        "hash": sha256_file(dex_file),
                        "from_bundle": False,
                    }
                )
                count += 1
            else:
                records.append(
                    {
                        "run_name": run_name,
                        "dex_name": dex_file.name,
                        "hash": sha256_file(dex_file),
                        "from_bundle": False,
                    }
                )
                count += 1

    # Handle the special case for classes1000.dex (which is in a different folder)
    classes_1000_dex = (
        run_dir
        / "build/intermediates/desugar_lib_dex/playProdRelease/l8DexDesugarLibPlayProdRelease/classes1000.dex"
    )
    if classes_1000_dex.exists():
        # Explicitly set classes1000.dex to classes2.dex
        records.append(
            {
                "run_name": run_name,
                "dex_name": "classes2.dex",  # Remap to classes2.dex explicitly
                "hash": sha256_file(classes_1000_dex),
                "from_bundle": False,
            }
        )


# Step 4: Create dataframe
df = pd.DataFrame(records)
df.set_index(["run_name", "from_bundle"], inplace=True)
df.sort_index(inplace=True)

In [19]:
df.loc["signal_7_41_3__dfs-sort__run_5"]

,dex_name,hash
from_bundle,,
False,classes.dex,ab85b85150775d1b4bf7f1f88fbf7f8bab9f051a7faa99...
False,classes3.dex,6bced746ee85eac654b2fcb64c2950695993abf0704275...
False,classes4.dex,b10f00c2d4d39f7e25c4bea7a807d5794472256625e778...
False,classes5.dex,c4a3f030950646989feb0df8aab55dadcf085c509796ab...
False,classes6.dex,e3fc2ead7fe95e874d3865e4f69e3026b2cc5cb76a691e...
False,classes7.dex,9834e7af7b38acd580bd91d0e3ef543f65a46fb0988aba...
False,classes2.dex,55cb98b238d8d8a1e12ebe1a30b86d477f055c7b8aa176...
True,classes.dex,ab85b85150775d1b4bf7f1f88fbf7f8bab9f051a7faa99...
True,classes2.dex,55cb98b238d8d8a1e12ebe1a30b86d477f055c7b8aa176...


Consistency check: build and bundle

In [20]:
for run_name, group in df.groupby(level="run_name"):
    try:
        # Split into bundle and non-bundle
        bundle_df = group.loc[(run_name, True)].set_index("dex_name")
        build_df = group.loc[(run_name, False)].set_index("dex_name")
        # Make sure both have the same dex files
        assert set(bundle_df.index) == set(
            build_df.index
        ), f"[{run_name}] Mismatched dex files"

        # Compare hashes
        for dex_name in bundle_df.index:
            hash_bundle = bundle_df.loc[dex_name, "hash"]
            hash_build = build_df.loc[dex_name, "hash"]
            if hash_bundle != hash_build:
                print(
                    f"[{run_name}] Hash mismatch in {dex_name} (build: {hash_build[:8]} / bundle: {hash_bundle[:8]})"
                )
    except KeyError:
        print(f"[{run_name}] Missing either bundle or build entries — skipping.")

[signal_7_41_3__dfs-sort__run_0] Missing either bundle or build entries — skipping.


In [21]:
print("Reproducibility check: bundle outputs across runs")

# Filter to just bundle outputs
bundle_df = df.xs(True, level="from_bundle")  # get only from_bundle == True

# Create a dictionary of {run_name: {dex_name: hash}}
bundle_hashes = {}

for run_name, group in bundle_df.groupby("run_name"):
    hash_map = group.set_index("dex_name")["hash"].to_dict()
    bundle_hashes[run_name] = hash_map

# Use first run as reference
reference_run, reference_map = next(iter(bundle_hashes.items()))

# Compare all other runs against the reference
for run_name, hash_map in bundle_hashes.items():
    if run_name == reference_run:
        continue

    if set(hash_map.keys()) != set(reference_map.keys()):
        print(f"[{run_name}] Mismatched dex file names")
        continue

    for dex_name in hash_map:
        if hash_map[dex_name] != reference_map[dex_name]:
            print(
                f"[{run_name}] Hash mismatch in {dex_name} (expected: {reference_map[dex_name][:8]}, got: {hash_map[dex_name][:8]})"
            )

Reproducibility check: bundle outputs across runs


> So, we can see that the 10 runs are repeatable

# Comparison with `play`

In [22]:
local = (
    df.loc[
        (
            df.index.get_level_values("run_name")[
                df.index.get_level_values("from_bundle") == True
            ].unique()[0],
            True,
        )
    ]
    .set_index("dex_name")["hash"]
    .to_dict()
)
local

{'classes.dex': 'ab85b85150775d1b4bf7f1f88fbf7f8bab9f051a7faa9948ec1bf9701f5c9ef6',
 'classes2.dex': '55cb98b238d8d8a1e12ebe1a30b86d477f055c7b8aa1769d986d38288dad5780',
 'classes3.dex': '6bced746ee85eac654b2fcb64c2950695993abf07042752343e98ee452b92fd3',
 'classes4.dex': 'b10f00c2d4d39f7e25c4bea7a807d5794472256625e77866aaf506e75ad8d7d9',
 'classes5.dex': 'c4a3f030950646989feb0df8aab55dadcf085c509796abfe46f1d037e2eef052',
 'classes6.dex': 'e3fc2ead7fe95e874d3865e4f69e3026b2cc5cb76a691ea7cf3f600930d74a9e',
 'classes7.dex': '9834e7af7b38acd580bd91d0e3ef543f65a46fb0988abac1247c9aae0253fcd4'}

In [23]:
play_folder = Path("./packer/play").resolve()
play = {}
for dex_file in play_folder.rglob("*.dex"):
    play[dex_file.name] = sha256_file(dex_file)
play

{'classes.dex': 'ab85b85150775d1b4bf7f1f88fbf7f8bab9f051a7faa9948ec1bf9701f5c9ef6',
 'classes2.dex': '55cb98b238d8d8a1e12ebe1a30b86d477f055c7b8aa1769d986d38288dad5780',
 'classes3.dex': '6bced746ee85eac654b2fcb64c2950695993abf07042752343e98ee452b92fd3',
 'classes4.dex': 'b10f00c2d4d39f7e25c4bea7a807d5794472256625e77866aaf506e75ad8d7d9',
 'classes5.dex': 'c4a3f030950646989feb0df8aab55dadcf085c509796abfe46f1d037e2eef052',
 'classes6.dex': 'e3fc2ead7fe95e874d3865e4f69e3026b2cc5cb76a691ea7cf3f600930d74a9e',
 'classes7.dex': '9834e7af7b38acd580bd91d0e3ef543f65a46fb0988abac1247c9aae0253fcd4'}

In [24]:
assert local == play, "Mismatch"

In [25]:
dfs_chaos_folder = Path(
    "./packer/results/signal_7_41_3__dfs-sort__chaos_1/dex/base/dex"
).resolve()
chaos = {}
for dex_file in play_folder.rglob("*.dex"):
    chaos[dex_file.name] = sha256_file(dex_file)
chaos

{'classes.dex': 'ab85b85150775d1b4bf7f1f88fbf7f8bab9f051a7faa9948ec1bf9701f5c9ef6',
 'classes2.dex': '55cb98b238d8d8a1e12ebe1a30b86d477f055c7b8aa1769d986d38288dad5780',
 'classes3.dex': '6bced746ee85eac654b2fcb64c2950695993abf07042752343e98ee452b92fd3',
 'classes4.dex': 'b10f00c2d4d39f7e25c4bea7a807d5794472256625e77866aaf506e75ad8d7d9',
 'classes5.dex': 'c4a3f030950646989feb0df8aab55dadcf085c509796abfe46f1d037e2eef052',
 'classes6.dex': 'e3fc2ead7fe95e874d3865e4f69e3026b2cc5cb76a691ea7cf3f600930d74a9e',
 'classes7.dex': '9834e7af7b38acd580bd91d0e3ef543f65a46fb0988abac1247c9aae0253fcd4'}

In [26]:
# C:\Users\Adrian\Code\reproducible-tests\packer\results\signal_7_41_3__dfs-sort__chaos_2\dex\base\dex
dfs_chaos_folder = Path(
    "./packer/results/signal_7_41_3__dfs-sort__chaos_2/dex/base/dex"
).resolve()
chaos2 = {}
for dex_file in play_folder.rglob("*.dex"):
    chaos2[dex_file.name] = sha256_file(dex_file)
chaos

{'classes.dex': 'ab85b85150775d1b4bf7f1f88fbf7f8bab9f051a7faa9948ec1bf9701f5c9ef6',
 'classes2.dex': '55cb98b238d8d8a1e12ebe1a30b86d477f055c7b8aa1769d986d38288dad5780',
 'classes3.dex': '6bced746ee85eac654b2fcb64c2950695993abf07042752343e98ee452b92fd3',
 'classes4.dex': 'b10f00c2d4d39f7e25c4bea7a807d5794472256625e77866aaf506e75ad8d7d9',
 'classes5.dex': 'c4a3f030950646989feb0df8aab55dadcf085c509796abfe46f1d037e2eef052',
 'classes6.dex': 'e3fc2ead7fe95e874d3865e4f69e3026b2cc5cb76a691ea7cf3f600930d74a9e',
 'classes7.dex': '9834e7af7b38acd580bd91d0e3ef543f65a46fb0988abac1247c9aae0253fcd4'}

In [27]:
chaos == chaos2 == play

True

## `v7.36.0` data

In [65]:
# Initialize list to collect all data
records = []

# Loop through each test run folder
for run_dir in root_dir.rglob("*7_36_0*"):
    if not run_dir.is_dir():
        continue

    run_name = run_dir.name
    aab_path = run_dir / "bundle.aab"
    build_aab = run_dir / build_aab_path
    bundle_dex_dir = run_dir / "dex"
    bundle_dex_dir.mkdir(exist_ok=True)

    if "ctime" in run_name:
        continue  # this was the case before

    # Step 1: Unzip .dex files into 'dex/' folder
    extract_dex_files(aab_path, bundle_dex_dir)

    # Step 2: Assert hash of bundle.aab matches the reference one
    if build_aab.exists():
        bundle_hash = sha256_file(aab_path)
        reference_hash = sha256_file(build_aab)
        assert bundle_hash == reference_hash, f"Hash mismatch in {run_name}"

    # Step 3: Hash each dex file in 'dex/' and in reference build
    # 3a: From bundle
    for dex_file in bundle_dex_dir.rglob("*.dex"):
        records.append(
            {
                "run_name": run_name,
                "dex_name": dex_file.name,
                "hash": sha256_file(dex_file),
                "from_bundle": True,
            }
        )

    # 3b: From build
    build_dex_folder = (
        run_dir / "build/intermediates/dex/playProdRelease/minifyPlayProdReleaseWithR8/"
    )
    count = 1
    for dex_file in build_dex_folder.rglob("*.dex"):
        if dex_file.exists():
            records.append(
                {
                    "run_name": run_name,
                    "dex_name": dex_file.name,
                    "hash": sha256_file(dex_file),
                    "from_bundle": False,
                }
            )
            count += 1
    classes_1000_dex = (
        run_dir
        / "build/intermediates/desugar_lib_dex/playProdRelease/l8DexDesugarLibPlayProdRelease/classes1000.dex"
    )
    if classes_1000_dex.exists():
        fname = f"classes{count}.dex"
        print(f"[INFO] Desugared libs dex exists, saving it as {fname}")
        records.append(
            {
                "run_name": run_name,
                "dex_name": fname,
                "hash": sha256_file(classes_1000_dex),
                "from_bundle": False,
            }
        )
# Step 4: Create dataframe
df36 = pd.DataFrame(records)
df36.set_index(["run_name", "from_bundle"], inplace=True)
df36.sort_index(inplace=True)

[INFO] Desugared libs dex exists, saving it as classes7.dex
[INFO] Desugared libs dex exists, saving it as classes7.dex
[INFO] Desugared libs dex exists, saving it as classes7.dex
[INFO] Desugared libs dex exists, saving it as classes7.dex
[INFO] Desugared libs dex exists, saving it as classes7.dex
[INFO] Desugared libs dex exists, saving it as classes7.dex


In [66]:
df36

dex_name  \
run_name                         from_bundle                 
signal_7_36_0__dfs-sort__chaos_1 False         classes.dex   
                                 False        classes2.dex   
                                 False        classes3.dex   
                                 False        classes4.dex   
                                 False        classes5.dex   
...                                                    ...   
signal_7_36_0__dfs-sort__run_3   True         classes3.dex   
                                 True         classes4.dex   
                                 True         classes5.dex   
                                 True         classes6.dex   
                                 True         classes7.dex   

                                                                                           hash  
run_name                         from_bundle                                                     
signal_7_36_0__dfs-sort__chaos_1 False        8488be4701f55c1caa9593995db3b4de928a8a047604e1...  
                                 False        2a8c03f6620548be3831349fccc78d233471b76209025f...  
                                 False        cde04edd26a0d79266b6b1c59dd350657c8b775f3f4e9a...  
                                 False        c9155026efe518cc350c806a24d1527d090f75db750df5...  
                                 False        3ddd5eb009fb37f284a91baa9e31e41294bd833c6d1529...  
...                                                                                         ...  
signal_7_36_0__dfs-sort__run_3   True         cde04edd26a0d79266b6b1c59dd350657c8b775f3f4e9a...  
                                 True         c9155026efe518cc350c806a24d1527d090f75db750df5...  
                                 True         3ddd5eb009fb37f284a91baa9e31e41294bd833c6d1529...  
                                 True         4d3a5ef6e9d7fd2b132808001c2b69da4989fcb8c50af1...  
                                 True         98059dae55a3c4c379e9a32f1b38b8a347bd6fb39ed87b...  

[84 rows x 2 columns]

In [67]:
print("Reproducibility check: bundle outputs across runs")

# Filter to just bundle outputs
bundle_df = df36.xs(True, level="from_bundle")  # get only from_bundle == True

# Create a dictionary of {run_name: {dex_name: hash}}
bundle_hashes = {}

for run_name, group in bundle_df.groupby("run_name"):
    hash_map = group.set_index("dex_name")["hash"].to_dict()
    bundle_hashes[run_name] = hash_map

# Use first run as reference
reference_run, reference_map = next(iter(bundle_hashes.items()))

# Compare all other runs against the reference
for run_name, hash_map in bundle_hashes.items():
    if run_name == reference_run:
        continue

    print(reference_run, run_name)

    if set(hash_map.keys()) != set(reference_map.keys()):
        print(f"[{run_name}] Mismatched dex file names")
        continue

    for dex_name in hash_map:
        if hash_map[dex_name] != reference_map[dex_name]:
            print(
                f"[{run_name}] Hash mismatch in {dex_name} (expected: {reference_map[dex_name][:8]}, got: {hash_map[dex_name][:8]})"
            )

Reproducibility check: bundle outputs across runs
signal_7_36_0__dfs-sort__chaos_1 signal_7_36_0__dfs-sort__chaos_2
[signal_7_36_0__dfs-sort__chaos_2] Hash mismatch in classes3.dex (expected: 3ddd5eb0, got: 2a8c03f6)
[signal_7_36_0__dfs-sort__chaos_2] Hash mismatch in classes5.dex (expected: 2a8c03f6, got: cde04edd)
[signal_7_36_0__dfs-sort__chaos_2] Hash mismatch in classes6.dex (expected: cde04edd, got: 3ddd5eb0)
signal_7_36_0__dfs-sort__chaos_1 signal_7_36_0__dfs-sort__chaos_3
[signal_7_36_0__dfs-sort__chaos_3] Hash mismatch in classes2.dex (expected: c9155026, got: 3ddd5eb0)
[signal_7_36_0__dfs-sort__chaos_3] Hash mismatch in classes3.dex (expected: 3ddd5eb0, got: c9155026)
[signal_7_36_0__dfs-sort__chaos_3] Hash mismatch in classes4.dex (expected: 4d3a5ef6, got: 2a8c03f6)
[signal_7_36_0__dfs-sort__chaos_3] Hash mismatch in classes5.dex (expected: 2a8c03f6, got: cde04edd)
[signal_7_36_0__dfs-sort__chaos_3] Hash mismatch in classes6.dex (expected: cde04edd, got: 4d3a5ef6)
signal_7_3

In [68]:
print("Reproducibility check: bundle outputs across runs (ignoring dex names)")

# Step 1: Slice to just from_bundle == True
df36_bundle = df36.xs(True, level="from_bundle")

# Step 2: Build a dictionary of {run_name: set of hashes, and track permutations of hash -> dex_name}
bundle_hash_sets = {}
hash_permutations = {}

for run_name, group in df36_bundle.groupby("run_name"):
    hash_set = set(group["hash"])
    bundle_hash_sets[run_name] = hash_set

    # Track which dex_name corresponds to which hash in each run
    for dex_name, hash_val in zip(group["dex_name"], group["hash"]):
        if hash_val not in hash_permutations:
            hash_permutations[hash_val] = set()
        hash_permutations[hash_val].add(dex_name)

# Step 3: Use the first run as the reference
reference_run, reference_hashes = next(iter(bundle_hash_sets.items()))

# Step 4: Compare all runs to the reference
any_mismatches = False  # Track if there were any mismatches
for run_name, hash_set in bundle_hash_sets.items():
    if run_name == reference_run:
        continue

    print(f"\nComparing {run_name} to reference {reference_run}:")

    # Track missing or extra hashes
    missing = reference_hashes - hash_set
    extra = hash_set - reference_hashes

    # If there are mismatches, report them
    if missing or extra:
        any_mismatches = True

        if missing:
            print(f"  Missing hashes: {[h[:8] for h in sorted(missing)]}")
            for m_hash in missing:
                print(
                    f"    Hash {m_hash[:8]} appears in dex_files: {sorted(hash_permutations[m_hash])}"
                )

        if extra:
            print(f"  Extra hashes:   {[h[:8] for h in sorted(extra)]}")
            for e_hash in extra:
                print(
                    f"    Hash {e_hash[:8]} appears in dex_files: {sorted(hash_permutations[e_hash])}"
                )

# Step 5: Final message if no mismatches were found
if not any_mismatches:
    print("\nAll runs have the same set of hashes. No mismatches found.")

Reproducibility check: bundle outputs across runs (ignoring dex names)

Comparing signal_7_36_0__dfs-sort__chaos_2 to reference signal_7_36_0__dfs-sort__chaos_1:

Comparing signal_7_36_0__dfs-sort__chaos_3 to reference signal_7_36_0__dfs-sort__chaos_1:

Comparing signal_7_36_0__dfs-sort__run_1 to reference signal_7_36_0__dfs-sort__chaos_1:

Comparing signal_7_36_0__dfs-sort__run_2 to reference signal_7_36_0__dfs-sort__chaos_1:

Comparing signal_7_36_0__dfs-sort__run_3 to reference signal_7_36_0__dfs-sort__chaos_1:

All runs have the same set of hashes. No mismatches found.


In [69]:
print("Reproducibility check: bundle outputs across runs (ignoring dex names)")

# Step 1: Slice to just from_bundle == True
df36_bundle = df36.xs(True, level="from_bundle")

# Step 2: Build a dictionary of {run_name: set of hashes} and track hash -> {dex_name: {run_names}} permutations
bundle_hash_sets = {}
hash_permutations = {}

for run_name, group in df36_bundle.groupby("run_name"):
    hash_set = set(group["hash"])
    bundle_hash_sets[run_name] = hash_set

    # Track which (dex_name -> set of run_names) corresponds to each hash
    for dex_name, hash_val in zip(group["dex_name"], group["hash"]):
        if hash_val not in hash_permutations:
            hash_permutations[hash_val] = (
                {}
            )  # Initialize as an empty dictionary for this hash
        if dex_name not in hash_permutations[hash_val]:
            hash_permutations[hash_val][
                dex_name
            ] = set()  # Initialize set for this dex_name

        hash_permutations[hash_val][dex_name].add(
            run_name
        )  # Add run_name to the set for this dex_name

# Step 3: Use the first run as the reference
reference_run, reference_hashes = next(iter(bundle_hash_sets.items()))

# Step 4: Compare all runs to the reference
any_mismatches = False  # Track if there were any mismatches
for run_name, hash_set in bundle_hash_sets.items():
    if run_name == reference_run:
        continue

    print(f"\nComparing {run_name} to reference {reference_run}:")

    # Track missing or extra hashes
    missing = reference_hashes - hash_set
    extra = hash_set - reference_hashes

    # If there are mismatches, report them
    if missing or extra:
        any_mismatches = True

        if missing:
            print(f"  Missing hashes: {[h[:8] for h in sorted(missing)]}")
            for m_hash in missing:
                # Report which dex_name and run_name this missing hash appeared in
                print(f"    Hash {m_hash[:8]} appears in:")
                for dex_name, runs in sorted(hash_permutations[m_hash].items()):
                    print(f"      - {dex_name}: {sorted(runs)}")

        if extra:
            print(f"  Extra hashes:   {[h[:8] for h in sorted(extra)]}")
            for e_hash in extra:
                # Report which dex_name and run_name this extra hash appeared in
                print(f"    Hash {e_hash[:8]} appears in:")
                for dex_name, runs in sorted(hash_permutations[e_hash].items()):
                    print(f"      - {dex_name}: {sorted(runs)}")

# Step 5: Final message if no mismatches were found
if not any_mismatches:
    print("\nAll runs have the same set of hashes. No mismatches found.")

Reproducibility check: bundle outputs across runs (ignoring dex names)

Comparing signal_7_36_0__dfs-sort__chaos_2 to reference signal_7_36_0__dfs-sort__chaos_1:

Comparing signal_7_36_0__dfs-sort__chaos_3 to reference signal_7_36_0__dfs-sort__chaos_1:

Comparing signal_7_36_0__dfs-sort__run_1 to reference signal_7_36_0__dfs-sort__chaos_1:

Comparing signal_7_36_0__dfs-sort__run_2 to reference signal_7_36_0__dfs-sort__chaos_1:

Comparing signal_7_36_0__dfs-sort__run_3 to reference signal_7_36_0__dfs-sort__chaos_1:

All runs have the same set of hashes. No mismatches found.


In [70]:
hash_permutations

{'8488be4701f55c1caa9593995db3b4de928a8a047604e1ab99643cc856de95c2': {'classes.dex': {'signal_7_36_0__dfs-sort__chaos_1',
   'signal_7_36_0__dfs-sort__chaos_2',
   'signal_7_36_0__dfs-sort__chaos_3',
   'signal_7_36_0__dfs-sort__run_1',
   'signal_7_36_0__dfs-sort__run_2',
   'signal_7_36_0__dfs-sort__run_3'}},
 'c9155026efe518cc350c806a24d1527d090f75db750df522590c2df4f8c74c60': {'classes2.dex': {'signal_7_36_0__dfs-sort__chaos_1',
   'signal_7_36_0__dfs-sort__chaos_2'},
  'classes3.dex': {'signal_7_36_0__dfs-sort__chaos_3'},
  'classes4.dex': {'signal_7_36_0__dfs-sort__run_1',
   'signal_7_36_0__dfs-sort__run_2',
   'signal_7_36_0__dfs-sort__run_3'}},
 '3ddd5eb009fb37f284a91baa9e31e41294bd833c6d1529ac9ca7b9e3b8a7c23f': {'classes3.dex': {'signal_7_36_0__dfs-sort__chaos_1'},
  'classes6.dex': {'signal_7_36_0__dfs-sort__chaos_2'},
  'classes2.dex': {'signal_7_36_0__dfs-sort__chaos_3'},
  'classes5.dex': {'signal_7_36_0__dfs-sort__run_1',
   'signal_7_36_0__dfs-sort__run_2',
   'signal_7_

In [71]:
rows = []

# Loop over the hash_permutations dictionary
for hash_val, dex_data in hash_permutations.items():
    # Shorten the hash to the first 8 characters
    short_hash = hash_val[:8]

    # Loop over each dex file and its runs
    for dex_name, run_names in dex_data.items():
        # Create a list of shortened run names (after the last __)
        short_run_names = {
            run.split("__")[-1] for run in run_names
        }  # Remove the prefix and keep only the part after "__"

        # For each run, create a row with the (short_hash, dex_name, run_name)
        for run in short_run_names:
            if (short_hash, dex_name, run) != ("4d3a5ef6", "classes5.dex", "run_1"):
                rows.append((short_hash, dex_name, run))

print(rows)
# Step 2: Convert to DataFrame
temp_df = pd.DataFrame(rows, columns=["short_hash", "filename", "run_name"])

# Step 3: Group by (short_hash, filename) and aggregate the run_name values into a list
df_grouped = (
    temp_df.groupby(["short_hash", "filename"])["run_name"].apply(list).reset_index()
)

df_grouped.set_index(["short_hash", "filename"], inplace=True)
# Step 4: Display the DataFrame
df_grouped

[('8488be47', 'classes.dex', 'chaos_3'), ('8488be47', 'classes.dex', 'chaos_2'), ('8488be47', 'classes.dex', 'chaos_1'), ('8488be47', 'classes.dex', 'run_2'), ('8488be47', 'classes.dex', 'run_1'), ('8488be47', 'classes.dex', 'run_3'), ('c9155026', 'classes2.dex', 'chaos_2'), ('c9155026', 'classes2.dex', 'chaos_1'), ('c9155026', 'classes3.dex', 'chaos_3'), ('c9155026', 'classes4.dex', 'run_1'), ('c9155026', 'classes4.dex', 'run_3'), ('c9155026', 'classes4.dex', 'run_2'), ('3ddd5eb0', 'classes3.dex', 'chaos_1'), ('3ddd5eb0', 'classes6.dex', 'chaos_2'), ('3ddd5eb0', 'classes2.dex', 'chaos_3'), ('3ddd5eb0', 'classes5.dex', 'run_1'), ('3ddd5eb0', 'classes5.dex', 'run_3'), ('3ddd5eb0', 'classes5.dex', 'run_2'), ('4d3a5ef6', 'classes4.dex', 'chaos_2'), ('4d3a5ef6', 'classes4.dex', 'chaos_1'), ('4d3a5ef6', 'classes6.dex', 'chaos_3'), ('4d3a5ef6', 'classes6.dex', 'run_1'), ('4d3a5ef6', 'classes6.dex', 'run_3'), ('4d3a5ef6', 'classes6.dex', 'run_2'), ('2a8c03f6', 'classes5.dex', 'chaos_1'), ('2a

run_name
short_hash filename                                                      
2a8c03f6   classes2.dex                             [run_1, run_3, run_2]
           classes3.dex                                         [chaos_2]
           classes4.dex                                         [chaos_3]
           classes5.dex                                         [chaos_1]
3ddd5eb0   classes2.dex                                         [chaos_3]
           classes3.dex                                         [chaos_1]
           classes5.dex                             [run_1, run_3, run_2]
           classes6.dex                                         [chaos_2]
4d3a5ef6   classes4.dex                                [chaos_2, chaos_1]
           classes6.dex                    [chaos_3, run_1, run_3, run_2]
8488be47   classes.dex   [chaos_3, chaos_2, chaos_1, run_2, run_1, run_3]
98059dae   classes7.dex  [chaos_3, chaos_2, chaos_1, run_2, run_1, run_3]
c9155026   classes2.dex                                [chaos_2, chaos_1]
           classes3.dex                                         [chaos_3]
           classes4.dex                             [run_1, run_3, run_2]
cde04edd   classes3.dex                             [run_1, run_3, run_2]
           classes5.dex                                [chaos_3, chaos_2]
           classes6.dex                                         [chaos_1]

In [72]:
rows = []

# Loop over the hash_permutations dictionary
for hash_val, dex_data in hash_permutations.items():
    # Shorten the hash to the first 8 characters
    short_hash = hash_val[:8]

    # Loop over each dex file and its runs
    for dex_name, run_names in dex_data.items():
        # Create a list of shortened run names (after the last __)
        short_run_names = {
            run.split("__")[-1] for run in run_names
        }  # Remove the prefix and keep only the part after "__"

        # For each run, create a row with the (short_hash, dex_name, run_name)
        for run in short_run_names:
            rows.append((short_hash, dex_name, run))

# Step 2: Convert to DataFrame
temp_df = pd.DataFrame(rows, columns=["short_hash", "filename", "run_name"])

# Step 3: Determine the fixed "run_1" filenames as the ground truth
run_1_files = temp_df[temp_df["run_name"] == "run_1"][
    ["filename", "short_hash"]
].sort_values(by="filename")

# Create a mapping of filenames to hash values from run_1
run_1_mapping = dict(zip(run_1_files["filename"], run_1_files["short_hash"]))

# Step 4: Now, let's look at how the chaos runs map to run_1 filenames
result = []

# Dynamically find all chaos runs (based on the fact that they're labeled as chaos_1, chaos_2, ...)
chaos_runs = [run for run in temp_df["run_name"].unique() if run.startswith("chaos_")]

# For each chaos run, we will determine how the filenames permute
for chaos_run in chaos_runs:
    # Filter chaos run files
    chaos_files = temp_df[temp_df["run_name"] == chaos_run][
        ["filename", "short_hash"]
    ].sort_values(by="filename")

    # Create the mapping from chaos filenames to their corresponding hashes
    chaos_mapping = dict(zip(chaos_files["filename"], chaos_files["short_hash"]))

    # We will now compare chaos_mapping with run_1_mapping to see how filenames map
    permutation = {
        filename: chaos_mapping.get(filename, None) for filename in run_1_mapping
    }

    result.append((chaos_run, permutation))

# Step 5: Show the final output (4 tuples or more if there's more chaos runs)
# 1. Print the ground truth from run_1
ground_truth = tuple(run_1_mapping.keys())
print(f"Ground truth (run_1): {ground_truth}")

inverse_mapping = {v: k for k, v in run_1_mapping.items()}
print(run_1_mapping)
print(inverse_mapping)
# 2. For each chaos run, print how the filenames got permuted based on the hash
for chaos_run, perm in result:
    permuted_filenames = tuple(
        inverse_mapping[perm[filename]] if perm[filename] is not None else "MISSING"
        for filename in run_1_mapping
    )

    # Highlight the mismatched files (those that do not match the original filenames)
    mismatches = [
        filename
        for filename, mapped_filename in zip(run_1_mapping.keys(), permuted_filenames)
        if filename != mapped_filename and mapped_filename != "MISSING"
    ]

    if mismatches:
        print(f"\nPermutation for {chaos_run}: {permuted_filenames}")
    else:
        print(f"\nPermutation for {chaos_run}: {permuted_filenames}")

Ground truth (run_1): ('classes.dex', 'classes2.dex', 'classes3.dex', 'classes4.dex', 'classes5.dex', 'classes6.dex', 'classes7.dex')
{'classes.dex': '8488be47', 'classes2.dex': '2a8c03f6', 'classes3.dex': 'cde04edd', 'classes4.dex': 'c9155026', 'classes5.dex': '3ddd5eb0', 'classes6.dex': '4d3a5ef6', 'classes7.dex': '98059dae'}
{'8488be47': 'classes.dex', '2a8c03f6': 'classes2.dex', 'cde04edd': 'classes3.dex', 'c9155026': 'classes4.dex', '3ddd5eb0': 'classes5.dex', '4d3a5ef6': 'classes6.dex', '98059dae': 'classes7.dex'}

Permutation for chaos_3: ('classes.dex', 'classes5.dex', 'classes4.dex', 'classes2.dex', 'classes3.dex', 'classes6.dex', 'classes7.dex')

Permutation for chaos_2: ('classes.dex', 'classes4.dex', 'classes2.dex', 'classes6.dex', 'classes3.dex', 'classes5.dex', 'classes7.dex')

Permutation for chaos_1: ('classes.dex', 'classes4.dex', 'classes5.dex', 'classes6.dex', 'classes2.dex', 'classes3.dex', 'classes7.dex')
